# Пункты 2–3: `X@W4^T` (bf16×int4) Triton + сравнение с `X@W16^T` на размерах Llama-3.2-1B-Instruct

Формат W4 **совпадает** с [triton_int4_quant_pack_mvp](./triton_int4_quant_pack_mvp.ipynb): по строке весов один `float32` scale, два int4 в одном `uint8` (low nibble — чётный столбец K, high nibble — нечётный).

`Y = X @ W^T`, `X` — [M, K] bf16, `W` — [N, K] в float, после квантования храним `packed: [N, (K+1)//2] uint8` и `scales: [N] float32`.

In [14]:
import time

import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "Нужна CUDA для Triton"
device = "cuda"
torch.manual_seed(0)

print("torch:", torch.__version__)
print("triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
triton: 3.1.0
GPU: NVIDIA A100 80GB PCIe


## Вспомогательные: quantize+pack (как в MVP) и dequant-референс

In [15]:
@triton.jit
def _row_absmax_kernel(
    x_ptr, scales_ptr, stride_xm, stride_xn, M, N, BLOCK_N: tl.constexpr,
):
    row = tl.program_id(0)
    offs_n = tl.arange(0, BLOCK_N)
    mask = offs_n < N
    x = tl.load(x_ptr + row * stride_xm + offs_n * stride_xn, mask=mask, other=0.0)
    absmax = tl.max(tl.abs(x), axis=0)
    scale = absmax / 7.0
    scale = tl.where(scale > 0, scale, 1.0)
    tl.store(scales_ptr + row, scale)


@triton.jit
def _quant_pack_int4_kernel(
    x_ptr, scales_ptr, packed_ptr, stride_xm, stride_xn, stride_pm, stride_pn, M, N, BLOCK_PACKS: tl.constexpr,
):
    row = tl.program_id(0)
    pack_block = tl.program_id(1)
    offs_pack = pack_block * BLOCK_PACKS + tl.arange(0, BLOCK_PACKS)
    col0, col1 = 2 * offs_pack, 2 * offs_pack + 1
    mask0, mask1 = col0 < N, col1 < N
    x0 = tl.load(x_ptr + row * stride_xm + col0 * stride_xn, mask=mask0, other=0.0)
    x1 = tl.load(x_ptr + row * stride_xm + col1 * stride_xn, mask=mask1, other=0.0)
    scale = tl.load(scales_ptr + row)
    inv_scale = 1.0 / scale
    q0f, q1f = x0 * inv_scale, x1 * inv_scale
    q0f = tl.where(q0f >= 0, q0f + 0.5, q0f - 0.5)
    q1f = tl.where(q1f >= 0, q1f + 0.5, q1f - 0.5)
    q0, q1 = tl.cast(q0f, tl.int32), tl.cast(q1f, tl.int32)
    q0, q1 = tl.maximum(tl.minimum(q0, 7), -8), tl.maximum(tl.minimum(q1, 7), -8)
    q0u, q1u = tl.cast(q0 + 8, tl.uint8), tl.cast(q1 + 8, tl.uint8)
    packed = q0u | (q1u << 4)
    out_mask = offs_pack < ((N + 1) // 2)
    tl.store(packed_ptr + row * stride_pm + offs_pack * stride_pn, packed, mask=out_mask)


def quantize_bf16_to_int4_packed_torch(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Torch fallback for long rows (for example Llama down_proj with K=8192). Quantization is not timed."""
    assert x.is_cuda and x.ndim == 2 and x.dtype in (torch.bfloat16, torch.float16)
    xf = x.to(torch.float32)
    scales = xf.abs().amax(dim=1) / 7.0
    scales = torch.where(scales > 0, scales, torch.ones_like(scales))
    qf = xf / scales[:, None]
    q = torch.where(qf >= 0, qf + 0.5, qf - 0.5).to(torch.int32).clamp(-8, 7)
    q_u = (q + 8).to(torch.uint8)
    if q_u.shape[1] % 2:
        q_u = torch.nn.functional.pad(q_u, (0, 1))
    packed = q_u[:, 0::2] | (q_u[:, 1::2] << 4)
    return packed.contiguous(), scales.contiguous()


def quantize_bf16_to_int4_packed(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    assert x.is_cuda and x.ndim == 2 and x.dtype in (torch.bfloat16, torch.float16)
    M, N = x.shape
    BLOCK_N = triton.next_power_of_2(N)
    if BLOCK_N > 4096:
        return quantize_bf16_to_int4_packed_torch(x)
    packed_cols = (N + 1) // 2
    scales = torch.empty((M,), device=x.device, dtype=torch.float32)
    packed = torch.empty((M, packed_cols), device=x.device, dtype=torch.uint8)
    _row_absmax_kernel[(M,)](
        x, scales, x.stride(0), x.stride(1), M, N, BLOCK_N=BLOCK_N,
    )
    BLOCK_PACKS = 256
    _quant_pack_int4_kernel[(M, triton.cdiv(packed_cols, BLOCK_PACKS))](
        x, scales, packed, x.stride(0), x.stride(1), packed.stride(0), packed.stride(1), M, N, BLOCK_PACKS=BLOCK_PACKS,
    )
    return packed, scales


def unpack_int4_to_float32(packed: torch.Tensor, scales: torch.Tensor, k_full: int) -> torch.Tensor:
    low = (packed & 0x0F).to(torch.int16)
    high = ((packed >> 4) & 0x0F).to(torch.int16)
    q = torch.empty((packed.shape[0], packed.shape[1] * 2), device=packed.device, dtype=torch.int16)
    q[:, 0::2] = low
    q[:, 1::2] = high
    q = q[:, :k_full].to(torch.float32)
    return (q - 8.0) * scales[:, None]


def dequantize_int4_to_bf16(packed: torch.Tensor, scales: torch.Tensor, k_full: int) -> torch.Tensor:
    return unpack_int4_to_float32(packed, scales, k_full).to(torch.bfloat16)


## Kernel: `Y = X @ W4^T` (X bf16, W int4+row scales)

После сравнения с `efficientml` используем более устойчивый паттерн: на каждом K-тайле распаковываем `uint8` packs в плотный блок `W_q` формы `(BLOCK_N, BLOCK_K)` и делаем **один** `tl.dot(X, W_q.T)`. Так как scale один на всю строку W, умножаем накопленный `acc` на `scales[out]` после цикла по K. Это ближе к рабочему kernel коллег и избегает двух отдельных dot по even/odd колонкам.

In [16]:
@triton.jit
def _matmul_x16_w4t_kernel(
    x_ptr,
    w_ptr,
    s_ptr,
    y_ptr,
    M,
    N,
    K: tl.constexpr,
    n_pack: tl.constexpr,
    stride_xm,
    stride_xk,
    stride_wn,
    stride_wp,
    stride_ym,
    stride_yn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    mask_m = offs_m < M
    mask_n = offs_n < N

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k_start in range(0, K, BLOCK_K):
        k_idx = k_start + offs_k
        pack_idx = k_idx // 2
        is_high = (k_idx % 2) == 1

        x = tl.load(
            x_ptr + tl.reshape(offs_m, (BLOCK_M, 1)) * stride_xm + tl.reshape(k_idx, (1, BLOCK_K)) * stride_xk,
            mask=tl.reshape(mask_m, (BLOCK_M, 1)) & (tl.reshape(k_idx, (1, BLOCK_K)) < K),
            other=0.0,
        )

        w_pack = tl.load(
            w_ptr + tl.reshape(offs_n, (BLOCK_N, 1)) * stride_wn + tl.reshape(pack_idx, (1, BLOCK_K)) * stride_wp,
            mask=tl.reshape(mask_n, (BLOCK_N, 1)) & (tl.reshape(pack_idx, (1, BLOCK_K)) < n_pack),
            other=0,
        )
        w_low = w_pack & 0x0F
        w_high = (w_pack >> 4) & 0x0F
        w_u = tl.where(tl.reshape(is_high, (1, BLOCK_K)), w_high, w_low)
        w_q = tl.cast(w_u, tl.int32) - 8
        w_q = tl.where(tl.reshape(k_idx, (1, BLOCK_K)) < K, w_q, 0)
        w = w_q.to(tl.bfloat16)

        acc += tl.dot(x, tl.trans(w))

    scales = tl.load(s_ptr + offs_n, mask=mask_n, other=0.0).to(tl.float32)
    acc = acc * tl.reshape(scales, (1, BLOCK_N))

    tl.store(
        y_ptr + tl.reshape(offs_m, (BLOCK_M, 1)) * stride_ym + tl.reshape(offs_n, (1, BLOCK_N)) * stride_yn,
        acc,
        mask=tl.reshape(mask_m, (BLOCK_M, 1)) & tl.reshape(mask_n, (1, BLOCK_N)),
    )


def matmul_x16_w4t(
    x: torch.Tensor,
    w_packed: torch.Tensor,
    scales: torch.Tensor,
) -> torch.Tensor:
    assert x.is_cuda and x.dtype == torch.bfloat16
    assert w_packed.is_cuda and w_packed.dtype == torch.uint8
    M, K = x.shape
    N, n_pack = w_packed.shape
    assert n_pack == (K + 1) // 2, f"packed last dim: {n_pack}, need {(K+1)//2} for K={K}"
    assert scales.shape == (N,) and scales.dtype == torch.float32
    y = torch.empty((M, N), device=x.device, dtype=torch.float32)
    if K <= 256:
        bm, bn, bk = 32, 32, 32
    elif K <= 2048:
        bm, bn, bk = 64, 64, 64
    else:
        bm, bn, bk = 64, 32, 128
    _matmul_x16_w4t_kernel[(triton.cdiv(M, bm), triton.cdiv(N, bn))](
        x,
        w_packed,
        scales,
        y,
        M,
        N,
        K,
        n_pack,
        x.stride(0),
        x.stride(1),
        w_packed.stride(0),
        w_packed.stride(1),
        y.stride(0),
        y.stride(1),
        BLOCK_M=bm,
        BLOCK_N=bn,
        BLOCK_K=bk,
    )
    return y


def matmul_x16_w4t_reference(x: torch.Tensor, w_packed: torch.Tensor, scales: torch.Tensor, k_in: int) -> torch.Tensor:
    w_up = unpack_int4_to_float32(w_packed, scales, k_in)
    return torch.matmul(x.to(torch.float32), w_up.t())


def matmul_x16_w16t_reference(x: torch.Tensor, w_bf16: torch.Tensor) -> torch.Tensor:
    return torch.matmul(x.to(torch.float32), w_bf16.to(torch.float32).t())


def matmul_x16_w16t_baseline(x: torch.Tensor, w_bf16: torch.Tensor) -> torch.Tensor:
    """bf16@bf16^T — типичный baseline для сравнения по времени (cuBLAS/Tensor Cores)."""
    return torch.matmul(x, w_bf16.t())

### Проверка: Triton vs полная деквантизация (fp32)

Ожидаем небольшое расхождение только из‑за порядка суммирования; `max |diff|` должен быть небольшим относительно масштаба выхода.

In [17]:
def run_correctness_test(m: int, n: int, k: int) -> None:
    w = torch.randn((n, k), device=device, dtype=torch.bfloat16) * 0.15
    wp, sc = quantize_bf16_to_int4_packed(w)
    x = torch.randn((m, k), device=device, dtype=torch.bfloat16) * 0.2
    y_ref = matmul_x16_w4t_reference(x, wp, sc, k)
    y_tr = matmul_x16_w4t(x, wp, sc)
    diff = (y_ref - y_tr).abs()
    r = y_ref.abs().max().clamp(min=1e-4)
    print(
        f"shape (M,N,K)=({m},{n},{k})  max|err|={diff.max().item():.4e}  "
        f"max|err|/max|ref|={(diff.max() / r).item():.3e}"
    )


run_correctness_test(64, 256, 128)
run_correctness_test(128, 2048, 2048)
run_correctness_test(512, 8192, 2048)

shape (M,N,K)=(64,256,128)  max|err|=1.7881e-07  max|err|/max|ref|=1.247e-07
shape (M,N,K)=(128,2048,2048)  max|err|=3.0994e-06  max|err|/max|ref|=4.565e-07
shape (M,N,K)=(512,8192,2048)  max|err|=1.6212e-05  max|err|/max|ref|=2.259e-06


### Размеры весов Llama-3.2-1B (`hidden_size=2048`, `intermediate_size=8192`, GQA: 8 KV-голов)

Для `F.linear` вес `W` имеет форму `(out_features, in_features)`. Ниже — характерные `N×K` для сравнения `X@W^T` при `M ∈ {128, 512, 2048}` (строки `X` — токены / batch*seq).

In [18]:
# См. https://huggingface.co/unsloth/Llama-3.2-1B-Instruct/blob/main/config.json
HIDDEN, INTERMEDIATE, NUM_KV, HEAD_DIM = 2048, 8192, 8, 64
KV_DIM = NUM_KV * HEAD_DIM  # 512

# (name, N_out, K_in) — соответствует W.shape
LLAMA_LINEAR_SHAPES: list[tuple[str, int, int]] = [
    ("q_proj / o_proj", HIDDEN, HIDDEN),
    ("k_proj or v_proj", KV_DIM, HIDDEN),
    ("gate_proj or up_proj", INTERMEDIATE, HIDDEN),
    ("down_proj", HIDDEN, INTERMEDIATE),
]
LLAMA_LINEAR_SHAPES

[('q_proj / o_proj', 2048, 2048),
 ('k_proj or v_proj', 512, 2048),
 ('gate_proj or up_proj', 8192, 2048),
 ('down_proj', 2048, 8192)]

In [19]:
@torch.inference_mode()
def bench_one(
    m: int,
    n: int,
    k: int,
    *,
    iters: int = 30,
    warmup: int = 10,
) -> tuple[float, float]:
    """Среднее время (ms) за вызов: (W4 Triton, W16 torch bf16). Квантизация не входит в замер — только matmul."""
    w_bf16 = torch.randn((n, k), device=device, dtype=torch.bfloat16) * 0.12
    packed, scales = quantize_bf16_to_int4_packed(w_bf16)
    x = torch.randn((m, k), device=device, dtype=torch.bfloat16) * 0.15
    for _ in range(warmup):
        _ = matmul_x16_w4t(x, packed, scales)
        _ = matmul_x16_w16t_baseline(x, w_bf16)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = matmul_x16_w4t(x, packed, scales)
    torch.cuda.synchronize()
    ms_w4 = (time.perf_counter() - t0) * 1000 / iters
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = matmul_x16_w16t_baseline(x, w_bf16)
    torch.cuda.synchronize()
    ms_w16 = (time.perf_counter() - t0) * 1000 / iters
    return ms_w4, ms_w16

In [20]:
TOKEN_BATCHES = (128, 512, 2048)

rows = []
for m in TOKEN_BATCHES:
    for name, n, k in LLAMA_LINEAR_SHAPES:
        ms4, ms16 = bench_one(m, n, k, iters=40, warmup=15)
        rows.append((m, name, n, k, ms4, ms16, ms4 / ms16))

print(f"{'M':>5}  {'layer':<22}  {'N':>5}  {'K':>5}   W4 ms   W16 ms   W4/W16")
for r in rows:
    m, name, n, k, ms4, ms16, ratio = r
    print(f"{m:5d}  {name:<22}  {n:5d}  {k:5d}   {ms4:6.3f}   {ms16:6.3f}   {ratio:5.2f}")

    M  layer                       N      K   W4 ms   W16 ms   W4/W16
  128  q_proj / o_proj          2048   2048    0.902    0.078   11.53
  128  k_proj or v_proj          512   2048    0.767    0.074   10.41
  128  gate_proj or up_proj     8192   2048    1.803    0.098   18.33
  128  down_proj                2048   8192    3.416    0.103   33.04
  512  q_proj / o_proj          2048   2048    1.804    0.095   18.90
  512  k_proj or v_proj          512   2048    0.769    0.077   10.03
  512  gate_proj or up_proj     8192   2048    5.966    0.316   18.89
  512  down_proj                2048   8192    8.624    0.223   38.60
 2048  q_proj / o_proj          2048   2048    5.880    0.313   18.79
 2048  k_proj or v_proj          512   2048    1.221    0.037   33.02
 2048  gate_proj or up_proj     8192   2048   22.085    0.688   32.10
 2048  down_proj                2048   8192   31.299    0.881   35.52
